In [13]:
from hierarchicalcausalmodels.models import HSCMParametric
from causalgraphicalmodels import CausalGraphicalModel

In [23]:
def collapse(HSCMParametric : HSCMParametric) -> CausalGraphicalModel:
    """
    Collapse a hierarchical structural causal model (HSCM) into a non-hierarchical CGM.

    This function takes an HSCMParametric object and collapses its hierarchical structure,
    resulting in a collapsed flat causal graphical model that retains the causal relationships of the original model.

    Parameters
    ----------
    HSCMParametric : HSCMParametric
        The hierarchical structural causal model to be collapsed.

    Returns
    -------
    CausalGraphicalModel
        A collapsed flat causal graphical model.
    """
    nodes = HSCMParametric.unit_nodes.copy()
    edges = HSCMParametric.edges.copy()
    for subunit in HSCMParametric.subunit_nodes:
        q_node = "Q" + subunit + "| pa(" + subunit + ")"
        nodes.add(q_node)
        for parent, child  in HSCMParametric.edges :
            if child == subunit and parent in nodes :
                edges.remove((parent, child))
                edges.add((parent, q_node))
            elif parent == subunit and child in nodes :
                edges.remove((parent, child))
                edges.add((q_node, child))
            elif parent == subunit:
                edges.remove((parent, child))
        collapsed_model = CausalGraphicalModel(nodes=list(nodes), edges=list(edges))
    return collapsed_model

In [ ]:
# Build three example HSCMs (confounder, confounder + interferer, instrument)
# and collapse each using the collapse() function.

def _empty_fun(*args, **kwargs):
    return None

# (a) Confounder: U -> A, U -> Y, A -> Y (A, Y are subunit-level)
hscm_confounder = HSCMParametric(
    nodes={"U", "A", "Y"},
    edges={("U", "A"), ("U", "Y"), ("A", "Y")},
    unit_nodes={"U"},
    subunit_nodes={"A", "Y"},
    sizes=[3],
    node_functions={"U": _empty_fun, "A": _empty_fun, "Y": _empty_fun},
    data={},
)
confounder_cgm = collapse(hscm_confounder)
print(confounder_cgm.dag.nodes)
print(confounder_cgm.dag.edges)

# (b) Confounder & Interferer: U -> A, U -> Y, A -> Y, A -> Z, Z -> Y
# A, Y are subunit-level; Z, U are unit-level.
hscm_confounder_interferer = HSCMParametric(
    nodes={"U", "Z", "A", "Y"},
    edges={("U", "A"), ("U", "Y"), ("A", "Y"), ("A", "Z"), ("Z", "Y")},
    unit_nodes={"U", "Z"},
    subunit_nodes={"A", "Y"},
    sizes=[3],
    node_functions={"U": _empty_fun, "Z": _empty_fun, "A": _empty_fun, "Y": _empty_fun},
    data={},
)
confounder_interferer_cgm = collapse(hscm_confounder_interferer)
print(confounder_interferer_cgm.dag.nodes)
print(confounder_interferer_cgm.dag.edges)


# (c) Instrument: U -> A, U -> Y, Z -> A, A -> Y
# Z, A are subunit-level; U, Y are unit-level.
hscm_instrument = HSCMParametric(
    nodes={"U", "Y", "Z", "A"},
    edges={("U", "A"), ("U", "Y"), ("Z", "A"), ("A", "Y")},
    unit_nodes={"U", "Y"},
    subunit_nodes={"Z", "A"},
    sizes=[3],
    node_functions={"U": _empty_fun, "Y": _empty_fun, "Z": _empty_fun, "A": _empty_fun},
    data={},
)
instrument_cgm = collapse(hscm_instrument)
print(instrument_cgm.dag.nodes)
print(instrument_cgm.dag.edges) 



['U', 'Q_A| pa(_A)', 'Q_Y| pa(_Y)']
[('U', 'Q_Y| pa(_Y)'), ('U', 'Q_A| pa(_A)')]
['U', 'Q_A| pa(_A)', 'Q_Y| pa(_Y)', 'Z']
[('U', 'Q_A| pa(_A)'), ('U', 'Q_Y| pa(_Y)'), ('Q_A| pa(_A)', 'Z'), ('Z', 'Q_Y| pa(_Y)')]
['U', 'Q_A| pa(_A)', 'Y', 'Q_Z| pa(_Z)']
[('U', 'Y'), ('U', 'Q_A| pa(_A)'), ('Q_A| pa(_A)', 'Y')]
